# Rohdaten ins Data Warehouse überführen

## Ziel
Dieses Notebook überführt die gecrawlten Rohdaten aus dem Data Lake in die SQLite-Datenbank `dwh_data.db`. Es erzeugt die Tabellen `newspapers` und `context` als Grundlage für alle folgenden Analysen.

## Vorgehen
- CSV-Dateien der Crawl-Tage chronologisch sammeln.
- Erfolgreich geladene HTML-Dateien einlesen.
- Klima-Kontexte extrahieren und in das Data Warehouse schreiben.
- Ergebnisdaten prüfen und optional als CSV exportieren.


In [ ]:
import os
import sys
import glob
import csv

# Projektbibliothek relativ zum Notebook einbinden
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
sys.path.append(os.path.join(notebook_dir, "..", "pylib"))

from handle_sqlite import read_table_as_dataframe
from handle_data_processing import batch_process_newspapers

db_path = os.path.join(notebook_dir, "..", "data_output", "dwh_data.db")
print(db_path)


## 1. Rohdaten-Dateien sammeln

Die Crawl-CSV-Dateien werden aus dem Data-Lake-Verzeichnis gelesen und für eine reproduzierbare Verarbeitung sortiert.


In [ ]:
# Use glob to list all CSV files in the specified directory that follow a date format in their names

csv_dir = os.path.join(notebook_dir, "..", "data_input", "data-lake")
csv_files = glob.glob(os.path.join(csv_dir, "*-*.csv"))


Die Dateien werden anhand des Datums im Dateinamen sortiert. Dadurch lässt sich der spätere Verarbeitungslauf zeitlich nachvollziehen.


In [ ]:
# we sort by the filename which contains the date, ignoring the directory path to make the sort efficient
csv_files.sort(key=lambda f: f.split('/')[-1])

# Output the total count of days (CSV files) to confirm the number of entries
print(f'Count of total days: {len(csv_files)}')


Aus den CSV-Dateien werden nur HTML-Pfade mit Statuscode 200 übernommen. Diese Seiten bilden die Rohbasis für die Extraktion der Klima-Kontexte.


In [ ]:
# Initialize an empty DataFrame to store newspaper data
newspapers = []

# Process each CSV file
for csv_file in csv_files:
    # Open and read the CSV file
    with open(csv_file, mode='r', encoding='utf-8') as file:
        reader = csv.DictReader(file)

        # For each row, check if the status code is 200 (OK) and append the newspaper data to the list
        for row in reader:
            if row['status'] == '200':
                newspapers.append({
                    'name': row['name'],
                    'date': row['date'],
                    'file_name': row['file_name'],
                    'encoding': row['encoding']
                })


Zur Kontrolle werden die ersten Einträge angezeigt, damit Struktur und Pfade vor der Verarbeitung plausibel sind.


In [ ]:
# Print the first two entries to verify the structure of the newspaper data
print(len(newspapers))
newspapers[:2]


## 2. Verarbeitung und Datenbankaufbau

Die Rohseiten werden im Batch verarbeitet. Die Ergebnisse werden in die DWH-Tabellen `newspapers` und `context` geschrieben.


In [ ]:
print(os.cpu_count())


In [ ]:
batch_process_newspapers(
    newspapers,
    batch_size=1024,
    num_workers=2,
    db_path=os.path.join(notebook_dir, "..", "data_output", "dwh_data.db"),
    input_path_prefix=os.path.join(notebook_dir, "..", "data_input")
)


## 3. Gespeicherte Daten prüfen

Nach dem Import werden die geschriebenen Tabellen aus der SQLite-Datenbank gelesen und stichprobenartig kontrolliert.


In [ ]:

# Read the 'newspapers' table from the database and display the first few rows
meta_data = read_table_as_dataframe("newspapers", db_path)
meta_data.info()


In [ ]:
# Read the 'context' table from the database and display the first few rows
context_data = read_table_as_dataframe("context", db_path)
context_data.head()


## 4. CSV-Export der verarbeiteten Daten (optional)

Die DWH-Tabellen können zusätzlich als CSV-Dateien mit Datumsstempel exportiert werden.


In [ ]:
import datetime

# Get today's date in YYYY-MM-DD format
today = datetime.datetime.now().strftime("%Y-%m-%d")

# Export the metadata and context data as CSV files, embedding today's date in the filenames
meta_data.to_csv(f"../data_output/dwh_meta_{today}.csv", index=False)
context_data.to_csv(f"../data_output/dwh_context_{today}.csv", index=False)


In [ ]:
# Check the number of unique newspaper names in the metadata (50 withou en)
# This helps verify that each newspaper is uniquely represented
meta_data.newspaper_name.nunique()


In [ ]:
# Check the number of unique publication dates in the metadata (must equal 'Count of total days': 1392)
# This ensures that we have distinct entries for each day the main page was crawled
meta_data.data_published.nunique()
